In [1]:
import numpy as np
import random

# --- Problem Definition & Data Generation ---
class VRPTWInstance:
    def __init__(self, num_customers=15, vehicle_capacity=50):
        np.random.seed(42)
        random.seed(42)
        self.num_customers = num_customers
        self.vehicle_capacity = vehicle_capacity

        # Depot at index 0, customers 1 to num_customers
        self.coords = np.random.randint(0, 100, size=(num_customers + 1, 2))
        self.demands = np.random.randint(5, 15, size=num_customers + 1)
        self.demands[0] = 0  # Depot demand is 0

        # Time windows [ReadyTime, DueTime] and Service Time
        self.time_windows = np.zeros((num_customers + 1, 2))
        self.service_times = np.zeros(num_customers + 1)
        for i in range(1, num_customers + 1):
            ready = random.randint(0, 100)
            due = ready + random.randint(30, 80)
            self.time_windows[i] = [ready, due]
            self.service_times[i] = random.randint(5, 15)

        # Depot time window is wide
        self.time_windows[0] = [0, 500]
        self.service_times[0] = 0

        # Distance matrix (Euclidean)
        self.dist_matrix = np.linalg.norm(self.coords[:, None, :] - self.coords[None, :, :], axis=-1)

# --- HCGA Components ---
class HCGA:
    def __init__(self, instance, pop_size=30, generations=50, mutation_rate=0.2):
        self.inst = instance
        self.pop_size = pop_size
        self.generations = generations
        self.mutation_rate = mutation_rate
        self.num_cust = instance.num_customers

    def constructive_heuristic(self, sequence):
        """Hybrid Step: Builds feasible routes from a sequence permutation."""
        routes = []
        current_route = []
        current_load = 0
        current_time = 0

        for customer in sequence:
            demand = self.inst.demands[customer]
            prev_node = current_route[-1] if current_route else 0
            travel_time = self.inst.dist_matrix[prev_node, customer]
            arrival_time = current_time + travel_time
            start_service = max(arrival_time, self.inst.time_windows[customer][0])

            # Check capacity and time window feasibility
            fits_capacity = current_load + demand <= self.inst.vehicle_capacity
            fits_time = start_service <= self.inst.time_windows[customer][1]

            # Check feasibility of returning to the depot
            return_time = start_service + self.inst.service_times[customer] + self.inst.dist_matrix[customer, 0]
            fits_depot_time = return_time <= self.inst.time_windows[0][1]

            if current_route and fits_capacity and fits_time and fits_depot_time:
                current_route.append(customer)
                current_load += demand
                current_time = start_service + self.inst.service_times[customer]
            else:
                if current_route:
                    routes.append(current_route)
                current_route = [customer]
                current_load = demand
                travel_from_depot = self.inst.dist_matrix[0, customer]
                current_time = max(travel_from_depot, self.inst.time_windows[customer][0]) + self.inst.service_times[customer]

        if current_route:
            routes.append(current_route)
        return routes

    def calculate_fitness(self, routes):
        """Total distance of all routes."""
        total_dist = 0
        for route in routes:
            if not route: continue
            total_dist += self.inst.dist_matrix[0, route[0]]
            for i in range(len(route) - 1):
                total_dist += self.inst.dist_matrix[route[i], route[i+1]]
            total_dist += self.inst.dist_matrix[route[-1], 0]
        return total_dist

    def crossover(self, parent1, parent2):
        """Order Crossover (OX1)."""
        size = len(parent1)
        start, end = sorted(random.sample(range(size), 2))
        child = [None] * size
        child[start:end] = parent1[start:end]

        pointer = end
        for item in parent2:
            if item not in child:
                if pointer >= size:
                    pointer = 0
                child[pointer] = item
                pointer += 1
        return child

    def mutate(self, sequence):
        """Swap mutation."""
        if random.random() < self.mutation_rate:
            idx1, idx2 = random.sample(range(len(sequence)), 2)
            sequence[idx1], sequence[idx2] = sequence[idx2], sequence[idx1]
        return sequence

    def solve(self):
        # Initialize Population (permutations of customer indices 1 to N)
        population = [random.sample(range(1, self.num_cust + 1), self.num_cust) for _ in range(self.pop_size)]
        best_seq = None
        best_routes = None
        best_cost = float('inf')

        for gen in range(self.generations):
            # Evaluate
            decoded_pop = [self.constructive_heuristic(ind) for ind in population]
            costs = [self.calculate_fitness(r) for r in decoded_pop]

            # Update Best
            min_idx = np.argmin(costs)
            if costs[min_idx] < best_cost:
                best_cost = costs[min_idx]
                best_seq = population[min_idx][:]
                best_routes = decoded_pop[min_idx]

            # Selection (Tournament)
            new_population = []
            for _ in range(self.pop_size):
                candidates = random.sample(list(zip(population, costs)), 3)
                winner = min(candidates, key=lambda x: x[1])[0]
                new_population.append(winner[:])

            # Crossover & Mutation
            next_gen = []
            for i in range(0, self.pop_size, 2):
                p1, p2 = new_population[i], new_population[min(i+1, self.pop_size-1)]
                c1 = self.mutate(self.crossover(p1, p2))
                c2 = self.mutate(self.crossover(p2, p1))
                next_gen.extend([c1, c2])

            population = next_gen[:self.pop_size]

        return best_routes, best_cost

# --- Execution and Output ---
instance = VRPTWInstance(num_customers=15, vehicle_capacity=50)
hcga = HCGA(instance, pop_size=40, generations=100, mutation_rate=0.25)
best_routes, best_cost = hcga.solve()

print("=== HCGA VRPTW Optimization Result ===")
print(f"Total Optimized Cost (Distance): {best_cost:.2f}")
print(f"Number of Vehicles/Routes required: {len(best_routes)}")
for idx, route in enumerate(best_routes):
    print(f"  Route {idx + 1}: Depot -> {' -> '.join(map(str, route))} -> Depot")


=== HCGA VRPTW Optimization Result ===
Total Optimized Cost (Distance): 813.20
Number of Vehicles/Routes required: 6
  Route 1: Depot -> 7 -> 9 -> Depot
  Route 2: Depot -> 14 -> 15 -> 4 -> Depot
  Route 3: Depot -> 6 -> Depot
  Route 4: Depot -> 12 -> 1 -> 8 -> 10 -> Depot
  Route 5: Depot -> 11 -> 13 -> 2 -> Depot
  Route 6: Depot -> 5 -> 3 -> Depot
